# Evaluation and unsuccessful tree experiments

Your original evaluate notebook was empty. This version completes the evaluation and records failed experiments without adding tree models to the submission path.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

search = [Path.cwd(), *Path.cwd().parents]
ROOT = next(candidate for parent in search for candidate in (parent, parent / 'deliver_ready')
            if (candidate / 'main.py').exists() and (candidate / 'src/workload/data.py').exists())
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'notebooks'))
from workload.data import load_data, split, validate_predictions

# Set VM_DATA_DIR, or change this variable to the directory with the two raw files.
candidates = [ROOT / 'challenge_data', ROOT.parent / 'forecasting-challenge/challenge_data']
DATA_DIR = Path(os.environ['VM_DATA_DIR']) if 'VM_DATA_DIR' in os.environ else next(
    (p for p in candidates if (p / 'vm_cpu_readings.csv.gz').exists() or (p / 'vm_cpu_readings.csv').exists()), candidates[0])
data = load_data(DATA_DIR)
n, future_times = split(data)
RESULTS = ROOT / 'src/results'
OUT = ROOT / 'notebooks/outputs'
OUT.mkdir(parents=True, exist_ok=True)
print(f'{n} training timestamps; {len(future_times)} forecast timestamps; raw files: {DATA_DIR}')


## Verify the current raw-data run

Run train.ipynb or the documented main command first. Metrics are read from explicitly saved files, not another notebook namespace.

In [ ]:
import json
from hashlib import sha256
if not (RESULTS / 'run_manifest.json').exists():
    raise FileNotFoundError('Run train.ipynb or main.py --evaluate first')
manifest = json.loads((RESULTS / 'run_manifest.json').read_text())
assert manifest['training_cpu_sha256'] == sha256(data.cpu.iloc[:n].to_numpy().tobytes()).hexdigest(), 'Results refer to different training data; rerun main.'
validate_predictions(pd.read_csv(ROOT / 'predictions.csv'), future_times)
validation = pd.read_csv(RESULTS / 'validation_predictions.csv')
holdout = pd.read_csv(RESULTS / 'holdout_predictions.csv')
from workload.evaluation import metrics
comparison = []
for phase, frame in [('validation', validation), ('retrospective_holdout', holdout)]:
    for model, part in frame.loc[frame.target.eq('aggregate_cpu_utilization')].groupby('model'):
        comparison.append({'phase': phase, 'model': model, **metrics(part.actual, part.prediction)})
display(pd.DataFrame(comparison))
display(Image(filename=str(RESULTS / 'figures/forecast_errors.png')))

## Failed Random Forest / GBM settings, with GAM as a comparator

Each method fitted nine independent group models, using lagged CPU, rolling statistics, relative time and 12/24 cyclic terms. Models predicted corrections to a 12-step seasonal anchor; all 24 leads used their own previous predictions. Only 38/62 supervised rows per group were available in validation, and 86 in the final fit. These particular settings lost to simple baselines; this does not prove all tree models are unsuitable. GAM is additive, not tree-based.

**My observations and explanation:**

_Add your notes here._

In [ ]:
RERUN_TREE_EXPERIMENTS = False  # Optional: install requirements-notebooks.txt, then set True.
if RERUN_TREE_EXPERIMENTS:
    from tree_experiments import run_experiments
    tree_metrics = run_experiments(data)
    tree_metrics.to_csv(OUT / 'tree_model_metrics_rerun.csv', index=False)
else:
    recorded_settings = json.loads((ROOT / 'notebooks/evidence/tree_model_settings.json').read_text())
    assert recorded_settings['training_cpu_sha256'] == manifest['training_cpu_sha256'], 'Recorded tree benchmarks refer to different training data; use an explicit rerun for the official 134-step window.'
    tree_metrics = pd.read_csv(ROOT / 'notebooks/evidence/tree_model_metrics.csv')
fleet_tree = tree_metrics.loc[tree_metrics.level.eq('fleet')]
display(fleet_tree.groupby(['phase', 'model'])[['MAE', 'RMSE']].mean())
print('Recorded settings:')
display(json.loads((ROOT / 'notebooks/evidence/tree_model_settings.json').read_text())['model_parameters'])

## Failure hypotheses and limitations

Few group-level samples, weakly stable cycles, and compounded recursive corrections may explain the poor results; they were not isolated experimentally. Averaging across VMs can favor simple time-series models while concealing individual errors. Earlier row-level LightGBM notebook scores are not promoted as clean evidence: some cells used summaries derived with an unsafe time cutoff or a different rollout protocol. Compare only like-for-like targets and validation windows.

The original SARIMA-24 trend was not the strongest earlier validation method. Its attractive final-window score is retrospective because the public holdout was inspected during exploration.

**My observations and explanation:**

_Add your notes here._